# RL SAE
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1.0.0/cookbook/notebooks/rl_sae.ipynb)

Use one SAE feature signature as a GRPO reward, then compare feature coverage before and after training.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower. This workflow loads both a trainable model and a frozen SAE host; it may exceed the memory of some Colab runtimes.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "idiom @ git+https://github.com/rotskoff-group/idiom.git@v1.0.0",
    ])


In [2]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record
from idiom.utils.notebook_helpers import save_run, train_and_save, training_config, write_fasta

started = time.perf_counter()

## Settings
Use a bundled signature or the `signature.json` from [feature enrichment](enriched_sae_features.ipynb). Set `CASE` and `SIGNATURE_NAME` to match that file, and use the same SAE weights.

Optionally set `ADDITIONAL_SIGNATURE` to another name in the same case. Training targets their union; the final comparison also reports each component’s coverage.


In [3]:
MODEL_ID = "jxliu2/idiom-300M"
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 1 # Prompts per training batch
SEED = 0
MAX_STEPS = 10 # Training steps for this demonstration
TARGET_LENGTH = 50 # Target IDR length in residues
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
N = 8 # Sequences sampled from each model for comparison
OUT_DIR = Path("rl_sae_outputs") / time.strftime("%Y%m%d-%H%M%S")

SAE_ID = "jxliu2/idiomsae-300M-L18-k32" # Pretrained SAE and its frozen host model
SAE_DEVICE = "cpu" # CPU scoring saves GPU memory; use "cuda" if memory permits
SIGNATURE_FILE = None # Default: bundled signatures; or a signature.json from the enriched SAE features notebook
CASE = "top30" # Signature group in the JSON file
SIGNATURE_NAME = "nucleolus" # Target signature within CASE
ADDITIONAL_SIGNATURE = None # Optional second signature name; train on the union

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Select the SAE reward
The reward is the fraction of target features that fire anywhere in an IDR, measured by a frozen pretrained host model and SAE. A union counts shared features once.


In [5]:
from idiom.sae.features import combine_signatures, load_signatures, write_signature
from idiom.train.grpo.reward.sae_feature import DEFAULT_FEATURES

sets = load_signatures(SIGNATURE_FILE or DEFAULT_FEATURES, case=CASE, sae=SAE_ID)
component_names = list(dict.fromkeys(
    [SIGNATURE_NAME] + ([ADDITIONAL_SIGNATURE] if ADDITIONAL_SIGNATURE else [])
))
missing = [name for name in component_names if name not in sets]
if missing:
    raise ValueError(f"Unknown signatures: {missing}. Available names: {sorted(sets)}")
targets = {name: sets[name] for name in component_names}
target_name = SIGNATURE_NAME
if ADDITIONAL_SIGNATURE:
    if "combined" in targets:
        raise ValueError("The name 'combined' is reserved for the union.")
    targets["combined"] = combine_signatures(targets)
    target_name = "combined"

training_signatures = write_signature(
    OUT_DIR / "signature.json", targets, case=CASE, provenance=dict(sae=SAE_ID),
)
reward_spec = dict(
    name="sae_signature", signature=target_name, features=str(training_signatures.resolve()),
    case=CASE, sae=SAE_ID, device=SAE_DEVICE,
)
display(pd.DataFrame({"signature": list(targets), "features": [len(ids) for ids in targets.values()]}))


,signature,features
0,nucleolus,30


## 2. Sample the starting model


In [6]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0, max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta(
    [Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
    OUT_DIR / "baseline.fasta",
)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Train with GRPO
Combine the target reward with length and amino-acid entropy penalties, then run a short demonstration. Each quadratic term is `-((value - target) / (0.2 * target))²`. Training saves a reloadable model and local logs in `OUT_DIR`.


In [7]:
cfg = training_config("grpo", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.trainer.enable_progress_bar = False
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = 2
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = 5e-6
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
terms = [
    dict(label="length", weight=1.0, reward=dict(name="length"),
         shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2)),
    dict(label="entropy", weight=1.0, reward=dict(name="entropy"),
         shaping=dict(name="quadratic", target=3.65, width=0.2)),
    dict(label="signature", weight=1.0, reward=reward_spec, shaping=dict(name="identity")),
]

cfg.reward.terms = terms
release = train_and_save(cfg)

[rank: 0] Seed set to 0


/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data2/scratch/jxliu2/idiom-public/.venv/lib/python3 ...
Using bfloat16 Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..


You are using a CUDA device ('NVIDIA H100 NVL') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:534: Found 270 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


`Trainer.fit` stopped: `max_steps=10` reached.


`weights_only` was not set, defaulting to `False`.


### Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [8]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "train/reward", "train/kl"] if name in metrics]
display(metrics[columns].tail(3))

,step,train/loss,train/reward,train/kl
7,7,0.454550,-11.898938,0.000225
8,8,-0.264365,-1.948248,0.000116
9,9,0.432840,-11.568149,0.000202


## 4. Generate from the trained model
Use the same sampling settings as the baseline for the comparison.


In [9]:
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Compare before and after training
Compare raw scores and the total shaped reward. Short demonstration runs may not improve the objective and do not establish sequence function. Full per-sequence scores are saved in `rewards.csv`.


In [10]:
from idiom.train.grpo.reward import build_reward

objective = build_reward(cfg.reward)
rows = []
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores["group"] = group
    rows.append(scores)
comparison = pd.concat(rows, ignore_index=True)
comparison.to_csv(OUT_DIR / "rewards.csv", index=False)
summary = comparison.groupby("group", sort=False).mean(numeric_only=True)
summary.to_csv(OUT_DIR / "reward_summary.csv")
score_columns = [f"{term['label']}_raw" for term in terms] + ["total_reward"]
display(summary[score_columns].round(3))
display(comparison[["group", "sequence", "total_reward"]].groupby("group", sort=False).head(2))

,length_raw,entropy_raw,signature_raw,total_reward
group,,,,
baseline,62.25,3.751,0.083,-9.026
adapted,57.75,3.630,0.033,-7.129


,group,sequence,total_reward
0,baseline,QATPAPQPNLDLSGTSTDMPESYEPSEHVVKKTPKQNRKRKLDPMP...,-20.760566
1,baseline,MCLDLPNLISGTRGRLKSTSEQPPPFLDRGKNSVNI,-1.993961
8,adapted,QATPAPQPNLDLSGTSTDMPESYEPSEHVVKKTPKQNRKTKLDPMP...,-21.060006
9,adapted,MCLDLPNLISGTRGRLKSTSEQPPPFLDRGKNSVNI,-1.993961


In [11]:
if ADDITIONAL_SIGNATURE:
    from idiom.train.grpo.reward.sae_feature import sae_signature

    coverage_rows = []
    for name in component_names + ["combined"]:
        score = sae_signature(
            name,
            features=str(training_signatures.resolve()),
            case=CASE,
            sae=SAE_ID,
            device=SAE_DEVICE,
        )
        for group, sequences in (("baseline", baseline), ("adapted", adapted)):
            coverage_rows.extend(
                dict(group=group, signature=name, sequence_id=i, coverage=value)
                for i, value in enumerate(score(sequences))
            )
    coverage = pd.DataFrame(coverage_rows)
    coverage.to_csv(OUT_DIR / "signature_coverage.csv", index=False)
    display(coverage.groupby(["group", "signature"]).coverage.mean().unstack())

## Results
`baseline.fasta`, `adapted.fasta`, and `rewards.csv` contain the before/after samples and scores. `signature.json` records the targets; `signature_coverage.csv` compares component coverage when signatures are combined. `model/` and `training/` contain the release, logs, and checkpoint.

Download results from Colab’s Files pane before the runtime ends. `run.json` records settings and package versions.


In [12]:
save_run(
    OUT_DIR,
    dict(model=MODEL_ID, device=DEVICE, seed=SEED, steps=MAX_STEPS, sampling=sampling),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: rl_sae_outputs/20260928-135352
